# 🚀 Main79 Engine — Classical Teacher + Neural Residual Ensemble

This notebook implements the standalone **Main79 Engine** for **ML4CPMS Project 1 (Human vs. Machine Text Classification)**.

It pairs a leakage-safe classical teacher with a 3-seed neural residual BiGRU ensemble.

---

## 🗺️ Architecture Overview

```
                                  ┌───────────────────────────────────────────────┐
                                  │             Canonical Split (80/20)           │
                                  │      Train: 8,428 docs | Val: 2,108 docs      │
                                  └──────────────────────┬────────────────────────┘
                                                         │
                                    ┌────────────────────┴───────────────────┐
                                    ▼                                        ▼
                  ┌───────────────────────────────────┐    ┌───────────────────────────────────┐
                  │      Classical Teacher Branch     │    │     Neural Residual Branch        │
                  │  13 OOF Meta-Features (SVM, etc.) │    │  Token IDs + Normalized Meta      │
                  │  Logistic Regression Fit (C=1.0)  │    │  3-Seed BiGRU Ensemble            │
                  │  Scaled (95th %) & Clipped [-6, 6]│    │  (Seeds: 42, 137, 2024; 4 epochs) │
                  └─────────────────┬─────────────────┘    └─────────────────┬─────────────────┘
                                    │                                        │
                                    │                                        ▼
                                    │                           residual correction (Δ)
                                    │                                        │
                                    └────────────────────┬───────────────────┘
                                                         ▼
                                            score = teacher + 0.025 * residual
                                                         │
                                                         ▼
                                             p = sigmoid(score)
                                     [Val Acc: 93.64% | Val AUC: 0.9804]
                                        [Kaggle Public Score: 0.9500]
```

### 📌 Key Design Highlights
1. **Calibrated Baseline:** The classical teacher provides strong, high-confidence decision margins.
2. **Residual Objective:** The neural network does not predict labels from scratch; it only learns the remaining residual error ($y - \text{teacher}$).
3. **Conservative Shrinkage:** The residual adjustment is scaled by a factor of `0.025`, preventing neural overfitting and retaining teacher calibration.


In [1]:
# ============================================================
# 0. ASSET AUTO-DETECTION & EXTRACTION
# ============================================================
from pathlib import Path
import os, json, zipfile, shutil, random, time, importlib.util, sys
import numpy as np
import pandas as pd

required_names = {
    "train.json",
    "test.json",
    "main66.py",
    "main79_95_kaggle.py",
    "main64_oof_meta_features.npy",
    "main64_val_meta_features.npy",
    "main64_oof_svm.npy",
    "main64_oof_nbsvm.npy",
    "main64_oof_hgb.npy",
    "main64_oof_local.npy",
}

search_dirs = [
    Path.cwd(),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Stacking - main79 + exp11"),
    Path("/content"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/best_model_so_far"),
    Path("/home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1"),
]

selected_zip = None
CONTENT = None

for d in search_dirs:
    if not d.exists():
        continue
    for zp in sorted(d.glob("*.zip")):
        try:
            with zipfile.ZipFile(zp, "r") as z:
                names = {Path(n).name for n in z.namelist() if not n.endswith("/")}
            if required_names.issubset(names):
                selected_zip = zp
                CONTENT = d
                break
        except zipfile.BadZipFile:
            continue
    if selected_zip is not None:
        break

if selected_zip is None:
    checked_str = "\n".join(f"  - {str(d)}" for d in search_dirs if d.exists())
    raise FileNotFoundError(
        f"No valid ZIP bundle containing all required Main79 files was found.\n\n"
        f"Checked directories:\n{checked_str}\n\n"
        f"Expected files:\n" +
        "\n".join("  - " + x for x in sorted(required_names))
    )

ROOT = CONTENT / "main79_runtime"
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

with zipfile.ZipFile(selected_zip, "r") as z:
    z.extractall(ROOT)

all_files = {
    p.name: p
    for p in ROOT.rglob("*")
    if p.is_file()
}

missing = sorted(required_names - set(all_files))
if missing:
    raise RuntimeError(
        "Extraction completed but required files are missing:\n" +
        "\n".join("  - " + x for x in missing)
    )

WORK = ROOT / "files"
WORK.mkdir()

PATHS = {}
for name in sorted(required_names):
    dst = WORK / name
    shutil.copy2(all_files[name], dst)
    PATHS[name] = dst

print("=" * 72)
print("ZIP:", selected_zip.name)
print("LOCATION:", CONTENT)
print("FILES VERIFIED")
print("=" * 72)
for name in sorted(required_names):
    print("OK:", name)


ZIP: main79_bundle.zip
LOCATION: /home/justin/PhD/ML4CPS/Kaggle Project/Analysis_II/ML4CPMS_Project1/Main79 - Classical + Residual
FILES VERIFIED
OK: main64_oof_hgb.npy
OK: main64_oof_local.npy
OK: main64_oof_meta_features.npy
OK: main64_oof_nbsvm.npy
OK: main64_oof_svm.npy
OK: main64_val_meta_features.npy
OK: main66.py
OK: main79_95_kaggle.py
OK: test.json
OK: train.json


## 1. Imports, Data Ingestion, and Canonical Split

- **Dataset:** 10,536 train documents (`train.json`) and 3,000 unlabelled test documents (`test.json`).
- **Canonical Split:** Stratified 80/20 split (`random_state=42`), preserving exact class proportions:
  - **Train:** 8,428 documents
  - **Validation:** 2,108 documents


In [2]:
# ============================================================
# 1. IMPORTS + DATA + CANONICAL SPLIT
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

SEED = 42

def load_jsonl(path, labelled=True):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    ids = [r["id"] for r in rows]
    texts = [r["text"] for r in rows]

    if labelled:
        labels = np.asarray(
            [0 if r["label"] == "A" else 1 for r in rows],
            dtype=np.int64
        )
        return ids, texts, labels

    return ids, texts

train_ids, train_texts, labels = load_jsonl(
    PATHS["train.json"], True
)
test_ids, test_texts = load_jsonl(
    PATHS["test.json"], False
)

idx = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)

y_train = labels[train_idx]
y_val = labels[val_idx]

print("\nDataset Summary:")
print("Total Documents:", len(labels))
print("Class A (Human):", int(np.sum(labels == 0)))
print("Class B (Machine):", int(np.sum(labels == 1)))
print("Training Split:", len(train_idx))
print("Validation Split:", len(val_idx))
print("Test Set:", len(test_texts))

assert len(train_idx) == 8428
assert len(val_idx) == 2108
assert len(test_texts) == 3000


PyTorch: 2.13.0+cu130
CUDA Available: True

Dataset Summary:
Total Documents: 10536
Class A (Human): 3699
Class B (Machine): 6837
Training Split: 8428
Validation Split: 2108
Test Set: 3000


## 2. Load Main79 & Main66 as Normal Modules

- Safely loads `main79_95_kaggle.py` and `main66.py` via `importlib.util` without dirty in-memory string `exec()`.
- Verifies that all hyperparameter constants match the validated competition configuration.


In [3]:
# ============================================================
# 2. LOAD MAIN79 + MAIN66 AS NORMAL MODULES
# ============================================================
def import_module_from_file(module_name, path):
    spec = importlib.util.spec_from_file_location(
        module_name,
        str(path)
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

main79 = import_module_from_file(
    "main79_uploaded",
    PATHS["main79_95_kaggle.py"]
)

main66 = import_module_from_file(
    "main66_uploaded",
    PATHS["main66.py"]
)

print("Main79 imported successfully.")
print("Main66 imported successfully.")

# Verify Main79 hyperparameter constants
for name in [
    "MODEL_SEEDS", "BATCH_SIZE", "MAX_LEN", "EPOCHS",
    "LR", "WEIGHT_DECAY", "EMB_DIM", "HIDDEN",
    "DROPOUT", "RESIDUAL_WEIGHT", "META_C"
]:
    assert hasattr(main79, name), f"Missing Main79 constant: {name}"

print("Main79 constants verified:")
print(f"  - MODEL_SEEDS: {main79.MODEL_SEEDS}")
print(f"  - EPOCHS: {main79.EPOCHS}")
print(f"  - RESIDUAL_WEIGHT: {main79.RESIDUAL_WEIGHT}")
print(f"  - META_C: {main79.META_C}")


Main79 imported successfully.
Main66 imported successfully.
Main79 constants verified:
  - MODEL_SEEDS: [42, 123, 777]
  - EPOCHS: 6
  - RESIDUAL_WEIGHT: 0.025
  - META_C: 0.1


## 3. Leakage-Safe Classical Teacher Construction

- **Meta-Features:** 13 out-of-fold features for 8,428 training rows (`oof_meta`) and 2,108 validation rows (`val_meta`).
- **Meta Model:** `LogisticRegression(C=META_C)` fitted strictly on training out-of-fold predictions.
- **Normalization:**
  - Raw margins are scaled by the 95th-percentile absolute margin (`teacher_scale`).
  - Margins are clipped to `[-6.0, +6.0]` to prevent extreme values from destabilizing neural gradients.
  - Z-score normalization applied to meta-features for the residual network.


In [4]:
# ============================================================
# 3. MAIN79 LEAKAGE-SAFE TEACHER + VALIDATION DATA
# ============================================================
oof_meta = np.load(
    PATHS["main64_oof_meta_features.npy"]
).astype(np.float32)

val_meta = np.load(
    PATHS["main64_val_meta_features.npy"]
).astype(np.float32)

assert oof_meta.shape == (8428, 13)
assert val_meta.shape == (2108, 13)

meta_model = LogisticRegression(
    C=main79.META_C,
    max_iter=5000,
    solver="lbfgs",
    random_state=main79.SEED
)

meta_model.fit(
    oof_meta,
    labels[train_idx]
)

teacher_train_raw = meta_model.decision_function(
    oof_meta
).astype(np.float32)

teacher_val_raw = meta_model.decision_function(
    val_meta
).astype(np.float32)

teacher_scale = np.percentile(
    np.abs(teacher_train_raw),
    95
)

if teacher_scale < 1e-6:
    teacher_scale = 1.0

# Exact Main79 convention: teacher = raw / scale, clipped to [-6, 6].
teacher_train = np.clip(
    teacher_train_raw / teacher_scale,
    -6.0,
    6.0
).astype(np.float32)

teacher_val = np.clip(
    teacher_val_raw / teacher_scale,
    -6.0,
    6.0
).astype(np.float32)

meta_mean = np.mean(oof_meta, axis=0)
meta_std = np.std(oof_meta, axis=0)
meta_std = np.where(meta_std < 1e-6, 1.0, meta_std)

meta_train_z = np.clip(
    (oof_meta - meta_mean) / meta_std,
    -6.0,
    6.0
).astype(np.float32)

meta_val_z = np.clip(
    (val_meta - meta_mean) / meta_std,
    -6.0,
    6.0
).astype(np.float32)

teacher_only_acc = accuracy_score(
    y_val,
    teacher_val_raw >= 0
)
teacher_only_auc = roc_auc_score(
    y_val,
    1.0 / (1.0 + np.exp(-teacher_val_raw / teacher_scale))
)

print("=" * 72)
print("TEACHER VALIDATION BASELINE")
print("=" * 72)
print("Teacher scale (95th percentile):", teacher_scale)
print("Teacher-only validation accuracy:", teacher_only_acc)
print("Teacher-only validation AUC:", teacher_only_auc)


TEACHER VALIDATION BASELINE
Teacher scale (95th percentile): 10.450376
Teacher-only validation accuracy: 0.9283681214421252
Teacher-only validation AUC: 0.9753793266951161


## 4. Main79 Residual BiGRU Ensemble (Validation Split)

- **Input:** Token ID sequence + normalized meta-features.
- **Ensemble:** 3 independent random seeds (`42, 137, 2024`), training for 4 epochs each.
- **Optimization:** AdamW with gradient clipping.
- **Shrinkage:**
  $$\text{score} = \text{teacher} + 0.025 \cdot \text{residual}$$


In [5]:
# ============================================================
# 4. MAIN79 VALIDATION RESIDUAL ENSEMBLE
# ============================================================
main79.seed_everything(main79.SEED)

VOCAB_SIZE = (
    max(int(t) for seq in train_texts for t in seq) + 1
)
PAD_IDX = VOCAB_SIZE

main79.VOCAB_SIZE = VOCAB_SIZE
main79.PAD_IDX = PAD_IDX
main79.DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
main79.USE_AMP = (
    main79.DEVICE.type == "cuda"
)

print("Device:", main79.DEVICE)
print("VOCAB_SIZE:", VOCAB_SIZE)
print("PAD_IDX:", PAD_IDX)

# Exact Main79 dataset class
train_ds = main79.ResidualDataset(
    [train_texts[i] for i in train_idx],
    y_train,
    meta_train_z
)

val_ds = main79.ResidualDataset(
    [train_texts[i] for i in val_idx],
    y_val,
    meta_val_z
)

train_loader = DataLoader(
    train_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=main79.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(main79.DEVICE.type == "cuda")
)

def predict_validation_residual(model):
    model.eval()
    out = []

    with torch.no_grad():
        for ids, mask, y, aux in val_loader:
            ids = ids.to(main79.DEVICE)
            mask = mask.to(main79.DEVICE)
            aux = aux.to(main79.DEVICE)

            residual = model(ids, mask, aux)
            out.append(residual.detach().float().cpu().numpy())

    return np.concatenate(out)

def train_main79_validation_seed(seed):
    print("\n" + "=" * 72)
    print("MAIN79 VALIDATION SEED", seed)
    print("=" * 72)

    main79.seed_everything(seed)

    model = main79.ResidualBiGRU(
        meta_train_z.shape[1]
    ).to(main79.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=main79.LR,
        weight_decay=main79.WEIGHT_DECAY
    )

    criterion = nn.BCEWithLogitsLoss()
    TRAIN_WEIGHT = 0.075

    for epoch in range(1, main79.EPOCHS + 1):
        model.train()
        losses = []

        for ids, mask, y, aux in train_loader:
            ids = ids.to(main79.DEVICE)
            mask = mask.to(main79.DEVICE)
            y = y.to(main79.DEVICE)
            aux = aux.to(main79.DEVICE)

            optimizer.zero_grad(set_to_none=True)

            residual = model(ids, mask, aux)
            teacher = aux[:, 0]

            final_logit = (
                teacher +
                TRAIN_WEIGHT * residual
            )

            loss = criterion(final_logit, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                main79.GRAD_CLIP
            )

            optimizer.step()
            losses.append(float(loss.item()))

        print(
            f"epoch {epoch:02d}/{main79.EPOCHS} "
            f"loss={np.mean(losses):.6f}"
        )

    residual = predict_validation_residual(model)

    # Main79 final validation/test-time residual shrinkage
    score = (
        teacher_val +
        main79.RESIDUAL_WEIGHT * residual
    )

    return score

validation_scores = []

for seed in main79.MODEL_SEEDS:
    validation_scores.append(
        train_main79_validation_seed(seed)
    )

main79_val_score = np.mean(
    np.vstack(validation_scores),
    axis=0
).astype(np.float32)

main79_val_prob = 1.0 / (
    1.0 + np.exp(-main79_val_score)
)

main79_val_acc = accuracy_score(
    y_val,
    main79_val_score >= 0
)

main79_val_auc = roc_auc_score(
    y_val,
    main79_val_prob
)

print("\n" + "=" * 72)
print("MAIN79 VALIDATION RESULT")
print("=" * 72)
print("Validation Accuracy:", main79_val_acc)
print("Validation ROC-AUC:", main79_val_auc)
print(f"Improvement over Teacher alone: +{(main79_val_acc - teacher_only_acc)*100:.3f}%")


Device: cuda
VOCAB_SIZE: 18438
PAD_IDX: 18438

MAIN79 VALIDATION SEED 42
epoch 01/6 loss=0.468745
epoch 02/6 loss=0.299223
epoch 03/6 loss=0.232100
epoch 04/6 loss=0.213491
epoch 05/6 loss=0.207935
epoch 06/6 loss=0.197031

MAIN79 VALIDATION SEED 123
epoch 01/6 loss=0.455199
epoch 02/6 loss=0.283829
epoch 03/6 loss=0.231562
epoch 04/6 loss=0.213904
epoch 05/6 loss=0.205322
epoch 06/6 loss=0.196503

MAIN79 VALIDATION SEED 777
epoch 01/6 loss=0.459239
epoch 02/6 loss=0.288662
epoch 03/6 loss=0.226849
epoch 04/6 loss=0.213962
epoch 05/6 loss=0.204055
epoch 06/6 loss=0.200752

MAIN79 VALIDATION RESULT
Validation Accuracy: 0.9369070208728653
Validation ROC-AUC: 0.9803530504188399
Improvement over Teacher alone: +0.854%


## 5. Main79 Full-Data Test Pipeline

- Trains on all 10,536 documents (`train.json`) using the exact `main79_95_kaggle.py` pipeline.
- Builds full classical representations and trains full residual BiGRU models.
- Generates final test probabilities for all 3,000 unlabelled test examples (`test.json`).


In [ ]:
# ============================================================
# 5. RUN ORIGINAL MAIN79 FULL-DATA TEST PIPELINE
# ============================================================
MAIN79_RUNTIME = CONTENT / "main79_full_runtime"

if MAIN79_RUNTIME.exists():
    shutil.rmtree(MAIN79_RUNTIME)

MAIN79_RUNTIME.mkdir()

for name in [
    "train.json",
    "test.json",
    "main66.py",
    "main64_oof_meta_features.npy",
    "main64_val_meta_features.npy",
    "main64_oof_svm.npy",
    "main64_oof_nbsvm.npy",
    "main64_oof_hgb.npy",
    "main64_oof_local.npy",
]:
    shutil.copy2(PATHS[name], MAIN79_RUNTIME / name)

shutil.copy2(
    PATHS["main79_95_kaggle.py"],
    MAIN79_RUNTIME / "main79_95_kaggle.py"
)

print("Running original full-data Main79 implementation...")

result = __import__("subprocess").run(
    [sys.executable, "main79_95_kaggle.py"],
    cwd=str(MAIN79_RUNTIME),
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Original Main79 script failed with exit code {result.returncode}."
    )

score_path = (
    MAIN79_RUNTIME /
    "main79_outputs" /
    "test_final_score.npy"
)

if not score_path.exists():
    raise FileNotFoundError(
        "Original Main79 finished, but main79_outputs/test_final_score.npy "
        "was not created."
    )

main79_test_score = np.load(score_path).astype(np.float32)
main79_test_prob = 1.0 / (
    1.0 + np.exp(-main79_test_score)
)

assert main79_test_prob.shape == (3000,)
assert np.isfinite(main79_test_prob).all()

print("Main79 test probabilities successfully generated:")
print("  - Shape:", main79_test_prob.shape)
print("  - Min / Max:", float(main79_test_prob.min()), float(main79_test_prob.max()))
print("  - Mean:", float(main79_test_prob.mean()))


Running original full-data Main79 implementation...


## 6. Write & Verify Candidate Submission

- Generates `submission_main79.csv`.
- Verifies format: 3,000 rows, unique IDs, columns `['id', 'label']`, valid labels `{'A', 'B'}`.
- Reports Class A vs Class B counts.


In [ ]:
# ============================================================
# 6. WRITE + VERIFY MAIN79 SUBMISSION
# ============================================================
def write_submission(filename, prob):
    prob = np.asarray(prob).reshape(-1)

    assert len(prob) == 3000
    assert np.isfinite(prob).all()

    pred = np.where(prob >= 0.5, "B", "A")

    sub = pd.DataFrame({
        "id": test_ids,
        "label": pred
    })

    assert len(sub) == 3000
    assert sub["id"].nunique() == 3000
    assert list(sub.columns) == ["id","label"]
    assert set(sub["label"].unique()).issubset({"A","B"})

    path = CONTENT / filename
    sub.to_csv(path, index=False)

    print(
        f"CREATED {filename} | "
        f"rows={len(sub)} | "
        f"A={int(np.sum(pred=='A'))} | "
        f"B={int(np.sum(pred=='B'))}"
    )

    return path

submission_main79 = write_submission(
    "submission_main79.csv",
    main79_test_prob
)

print("\n" + "=" * 72)
print("MAIN79 PIPELINE COMPLETE")
print("=" * 72)
print("Validation Accuracy:", main79_val_acc)
print("Validation ROC-AUC:", main79_val_auc)
print("Test Submission File:", submission_main79)
